# MindSafe: Primary Survey Data Preprocessing & Feature Engineering
**Project:** MindSafe — An AI-Based Cyberbullying & Its Impact on Mental Health  
**Study Type:** Primary Quantitative Survey Research  
**Pipeline Phase:** Phase 1 — Comprehensive Data Cleaning, Quality Audit, Target Leakage Prevention, Encoding, and Stratified Partitioning  

---

### Core Data Preprocessing Objectives:
1. **Provenance & Immutability:** The raw survey responses in `Data/raw/` remain 100% untouched and preserved.
2. **Justified Cleaning:** Address encoding nuances (Windows-1252 en-dashes), whitespace, and remove completely unattempted submissions ($N=2$) without discarding valid responses.
3. **Target Leakage Control:** Exclude symptom-overlapping variables (Q13 and Q14) from the ML predictor set to ensure models predict impact from external exposure and behavioral markers.
4. **Encoding Taxonomy:** Apply Ordinal, One-Hot, and Multi-Hot encoding based on questionnaire item measurement levels.
5. **Exact Feature Generation:** Produce exactly 54 numeric predictor features from the 13 retained survey items.
6. **Data Snooping Prevention:** Fit encoders and learn vocabulary exclusively on the 80% stratified training partition ($N=411$) while keeping the 20% test partition ($N=103$) unseen.

## 1. Load Raw Data
Locate the raw Google Forms survey export in `Data/raw/`, inspect its encoding and cryptographic checksum, and load it into pandas.

In [1]:
import os
import sys
import glob
import hashlib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

# Locate raw survey CSV
candidates = [
    os.path.join('..', 'Data', 'raw'),
    os.path.join('Data', 'raw'),
    os.path.join(os.getcwd(), 'Data', 'raw'),
    os.path.join(os.path.dirname(os.getcwd()), 'Data', 'raw')
]

raw_dir = None
for c in candidates:
    if os.path.exists(c) and glob.glob(os.path.join(c, '*.csv')):
        raw_dir = c
        break

if not raw_dir:
    raise FileNotFoundError("Could not find Data/raw directory with survey CSV.")

raw_csv_path = glob.glob(os.path.join(raw_dir, '*.csv'))[0]
print(f"Located raw survey CSV: {raw_csv_path}")

# Verify file provenance via SHA-256 hash
with open(raw_csv_path, 'rb') as f:
    raw_bytes = f.read()
raw_sha256 = hashlib.sha256(raw_bytes).hexdigest()
print(f"Raw File Size: {len(raw_bytes):,} bytes | SHA-256: {raw_sha256}")

# Safely decode Windows-1252 single-byte characters into pandas
try:
    df_raw = pd.read_csv(raw_csv_path, encoding='utf-8')
except UnicodeDecodeError:
    print("Direct UTF-8 decode failed due to single-byte en-dash (0x96). Reading with windows-1252 encoding.")
    df_raw = pd.read_csv(raw_csv_path, encoding='windows-1252')

print(f"Raw Dataset Dimensions: {df_raw.shape[0]} Rows × {df_raw.shape[1]} Columns\n")
print("First 3 records from raw survey data:")
display(df_raw.head(3))

Located raw survey CSV: ..\Data\raw\Survey on Cyberbullying, Mental Health, and Cyber Law Awareness Among Social Media Users (Responses) - Form Responses 1.csv
Raw File Size: 108,329 bytes | SHA-256: 090fa5883977de148bc01489b999f88ee02af7ed4132f7c53f785c61b25b46bf
Direct UTF-8 decode failed due to single-byte en-dash (0x96). Reading with windows-1252 encoding.
Raw Dataset Dimensions: 521 Rows × 19 Columns

First 3 records from raw survey data:


,Timestamp,[SECTION A: Basic Information]\r\n\r\n 1. What is your age?,2. What is your gender?,[SECTION B: Social Media Usage] \r\n\r\n3. Which social media platforms do you use regularly? (Select all that apply),4. How many hours do you spend on social media per day?,[SECTION C: Cyberbullying Experience] \r\n\r\n5. Have you personally experienced cyberbullying or online harassment on social media?,6. Have you ever witnessed someone being cyberbullied?,"7. Have you ever posted, shared, or sent a message online that could have hurt or offended someone?","8. If yes, what was the main reason for your action?",9. What type of cyberbullying have you experienced or observed? (Select all that apply),10. On which social media platform did the incident occur?,11. How often have you experienced cyberbullying or online harassment?,[SECTION D: Mental Health Impact] \r\n\r\n12. Do you think cyberbullying or online harassment affected your mental health?,13. Which of the following did you experience? (Select all that apply),"14. On a scale of 1â€“5, how severe was the emotional impact?",15. Did you seek help from anyone? (Select all that apply),"[SECTION E: Cyber Law Awareness and Prevention] \r\n\r\n16. If you did not report the incident, what was the main reason?",17. In which area did you experience cyberbullying or online harassment?,18. What action did you take after experiencing cyberbullying or online harassment? (Select all that apply)
0,09-09-2026 07:36,18–22,Female,"Instagram, WhatsApp, YouTube",3â€“5 hours,Yes,No,Yes,I was angry,"Offensive Comments, Body Shaming, Threats, Fak...",Instagram,Rarely,Not at all,"Stress, Anxiety, Depression",1 (Very Low),"Friends, Family, Teacher",Fear,Workplace,"Ignored it, Blocked the user"
1,09-09-2026 19:19,18–22,Male,"Instagram, WhatsApp, YouTube",3â€“5 hours,Yes,Yes,No,It was a joke,"Offensive Comments, Body Shaming, Hate Speech,...",Instagram,Rarely,Slightly,Anxiety,4,"Friends, Family, Helpline",Fear,Friends Circle,Blocked the user
2,09-09-2026 19:23,18–22,Male,WhatsApp,1â€“3 hours,Yes,Yes,Yes,Self-defense,"Threats, Sexual Harassment",WhatsApp,Sometimes,Slightly,Anger,2,"Family, Psychologist",Lack of Awareness,Friends Circle,Told Friends / Family


## 2. Data Cleaning
Perform rigorous, justified cleaning:
- Standardize en-dashes (`–` to `-` or normalized Unicode)
- Strip leading and trailing whitespace
- Normalize missing values and empty strings while preserving legitimate 'None' selections
- Filter out completely blank submissions (rows where all 18 survey questions were unattempted)

In [2]:
# 1. Clean string encoding: replace Windows-1252 special characters
df_cleaned = df_raw.copy()
for col in df_cleaned.columns:
    if df_cleaned[col].dtype == 'object':
        df_cleaned[col] = df_cleaned[col].astype(str).str.replace('', '–', regex=False)
        df_cleaned[col] = df_cleaned[col].str.strip()

# 2. Convert string 'nan' to np.nan while preserving 'None' options (e.g., in Q13)
col13_name = df_cleaned.columns[13]
for col in df_cleaned.columns:
    if col != col13_name:
        df_cleaned[col] = df_cleaned[col].replace({'nan': np.nan, '': np.nan})
    else:
        df_cleaned[col] = df_cleaned[col].apply(lambda x: np.nan if x in ['nan', ''] else x)

# 3. Detect and remove completely blank submissions (Rows 21 and 80)
survey_cols = df_cleaned.columns[1:]
blank_mask = df_cleaned[survey_cols].isna().sum(axis=1) == 18
blank_indices = df_cleaned[blank_mask].index.tolist()

print(f"Dataset Size Before Cleaning : {len(df_cleaned)} rows × {df_cleaned.shape[1]} columns")
print(f"Completely Blank Submissions : {len(blank_indices)} (Indices: {blank_indices})")

df_cleaned = df_cleaned.drop(index=blank_indices).reset_index(drop=True)
print(f"Dataset Size After Cleaning  : {len(df_cleaned)} rows × {df_cleaned.shape[1]} columns")
assert len(df_cleaned) == 519, "Analytical sample must contain exactly 519 records."
print("VERIFIED: Exactly 519 valid participant records retained.")

Dataset Size Before Cleaning : 521 rows × 19 columns
Completely Blank Submissions : 2 (Indices: [21, 80])
Dataset Size After Cleaning  : 519 rows × 19 columns
VERIFIED: Exactly 519 valid participant records retained.


## 3. Data Quality Report
Generate a standardized data quality audit summarizing each variable's data type, total responses, valid count, missing percentage, unique categories, and sample values.

In [3]:
quality_rows = []
for idx, col in enumerate(df_cleaned.columns):
    valid_cnt = df_cleaned[col].notna().sum()
    miss_cnt = df_cleaned[col].isna().sum()
    miss_pct = round(miss_cnt / len(df_cleaned) * 100, 2)
    unique_cnt = df_cleaned[col].nunique()
    sample_vals = str(list(df_cleaned[col].dropna().unique()[:3]))
    
    quality_rows.append({
        'Index': idx,
        'Column_Name': col,
        'Data_Type': str(df_cleaned[col].dtype),
        'Total_Responses': len(df_cleaned),
        'Valid_Responses': valid_cnt,
        'Missing_Count': miss_cnt,
        'Missing_Percentage (%)': miss_pct,
        'Unique_Categories': unique_cnt,
        'Sample_Categories': sample_vals
    })

df_quality_report = pd.DataFrame(quality_rows)
display(df_quality_report[['Index', 'Column_Name', 'Valid_Responses', 'Missing_Count', 'Missing_Percentage (%)', 'Unique_Categories', 'Sample_Categories']])

,Index,Column_Name,Valid_Responses,Missing_Count,Missing_Percentage (%),Unique_Categories,Sample_Categories
0,0,Timestamp,519,0,0.000,500,"['09-09-2026 07:36', '09-09-2026 19:19', '09-0..."
1,1,[SECTION A: Basic Information]\r\n\r\n 1. What...,519,0,0.000,6,"['18–22', '23–30', 'Above 30']"
2,2,2. What is your gender?,519,0,0.000,4,"['Female', 'Male', 'Prefer not to say']"
3,3,[SECTION B: Social Media Usage] \r\n\r\n3. Whi...,519,0,0.000,15,"['Instagram, WhatsApp, YouTube', 'WhatsApp', '..."
4,4,4. How many hours do you spend on social media...,519,0,0.000,4,"['3â€“5 hours', '1â€“3 hours', 'More than 5 ho..."
5,5,[SECTION C: Cyberbullying Experience] \r\n\r\n...,518,1,0.190,2,"['Yes', 'No']"
6,6,6. Have you ever witnessed someone being cyber...,519,0,0.000,2,"['No', 'Yes']"
7,7,"7. Have you ever posted, shared, or sent a mes...",519,0,0.000,3,"['Yes', 'No', 'Not Sure']"
8,8,"8. If yes, what was the main reason for your a...",147,372,71.680,8,"['I was angry', 'It was a joke', 'Self-defense']"
9,9,9. What type of cyberbullying have you experie...,511,8,1.540,40,"['Offensive Comments, Body Shaming, Threats, F..."


## 4. Target Variable Identification
Isolate the primary single prediction target: **Mental Health Impact** (`Question 12`).  
The target variable is partitioned into four distinct classes: `Not at all`, `Slightly`, `Moderately`, and `Severely`.

In [4]:
col_target = [c for c in df_cleaned.columns if '12. Do you think cyberbullying' in c][0]
valid_target = df_cleaned[col_target].dropna()
target_counts = valid_target.value_counts()
ordered_classes = ['Not at all', 'Slightly', 'Moderately', 'Severely']

target_summary_rows = []
for idx, cat in enumerate(ordered_classes):
    cnt = target_counts.get(cat, 0)
    target_summary_rows.append({
        'Class_Index': idx,
        'Mental_Health_Impact_Class': cat,
        'Frequency': cnt,
        'Valid_Percentage (%)': round(cnt / len(valid_target) * 100, 2),
        'Total_Percentage (%)': round(cnt / len(df_cleaned) * 100, 2)
    })

df_target_summary = pd.DataFrame(target_summary_rows)
display(df_target_summary)
print(f"Target Summary: Total = {len(df_cleaned)} | Valid = {len(valid_target)} | Missing = {df_cleaned[col_target].isna().sum()} (0.96%)")

,Class_Index,Mental_Health_Impact_Class,Frequency,Valid_Percentage (%),Total_Percentage (%)
0,0,Not at all,229,44.550,44.120
1,1,Slightly,137,26.650,26.400
2,2,Moderately,107,20.820,20.620
3,3,Severely,41,7.980,7.900


Target Summary: Total = 519 | Valid = 514 | Missing = 5 (0.96%)


## 5. Target Leakage Prevention
Rigorous methodology to prevent circular reasoning and shortcut learning in machine learning.

### Target vs Predictor Conceptual Separation:
- **TARGET ($y$):** `Mental Health Impact` (Self-reported impact level on psychological well-being).
- **PREDICTORS ($X$):** Objective behavioral, demographic, and contextual markers of cyberbullying exposure.

### Excluded Variables & Rationale:
1. **Question 13 (`Negative Emotional Symptoms`):** Asking respondents whether they felt depression, stress, anxiety, or anger directly measures their mental health state rather than external risk factors.
2. **Question 14 (`Emotional Impact Severity`):** Rating emotional trauma on a 1–5 scale tautologically duplicates the target impact outcome.
3. **Timestamp:** Metadata unrelated to participant behavior.
4. **Conditional Skip-Logic Questions (Q8 & Q16):** Highly sparse structural conditional fields (Perpetrator motivation and reason for not reporting).

Retaining Q13 and Q14 in production creates severe target leakage, causing models to rely on psychological symptom checklists rather than learning true cyberbullying exposure patterns.

In [5]:
print("""
================================================================================
CRITICAL TARGET LEAKAGE AUDIT: SCENARIO A vs SCENARIO B
================================================================================
Target Outcome: Mental_Health_Impact (Not at all, Slightly, Moderately, Severely)

EXCLUDED FROM MACHINE LEARNING PREDICTORS:
- [Q13] Which of the following did you experience? (Stress, Anxiety, Depression...)
- [Q14] On a scale of 1–5, how severe was the emotional impact?
- [Timestamp] Survey submission timestamp
- [Q8 & Q16] Structural skip-logic conditional responses (>70% non-applicable)

RETAINED ML PREDICTORS (13 High-Integrity Items):
1. Age (Demographic baseline)
2. Gender (Demographic grouping)
3. Social media platforms regularly used (Digital presence)
4. Daily usage hours (Screen engagement)
5. Personally experienced cyberbullying (Direct exposure)
6. Witnessed cyberbullying (Observed hostility)
7. Posted offensive content (Communicative friction)
8. Types of cyberbullying experienced/observed (Harassment typology)
9. Platform where incident occurred (Digital environment)
10. Frequency of cyberbullying encounters (Exposure intensity)
11. Support / help sought (Support system)
12. Area / context of harassment (Social setting)
13. Actions taken after incident (Protective behavior)
================================================================================
""")


CRITICAL TARGET LEAKAGE AUDIT: SCENARIO A vs SCENARIO B
Target Outcome: Mental_Health_Impact (Not at all, Slightly, Moderately, Severely)

EXCLUDED FROM MACHINE LEARNING PREDICTORS:
- [Q13] Which of the following did you experience? (Stress, Anxiety, Depression...)
- [Q14] On a scale of 1–5, how severe was the emotional impact?
- [Timestamp] Survey submission timestamp
- [Q8 & Q16] Structural skip-logic conditional responses (>70% non-applicable)

RETAINED ML PREDICTORS (13 High-Integrity Items):
1. Age (Demographic baseline)
2. Gender (Demographic grouping)
3. Social media platforms regularly used (Digital presence)
4. Daily usage hours (Screen engagement)
5. Personally experienced cyberbullying (Direct exposure)
6. Witnessed cyberbullying (Observed hostility)
7. Posted offensive content (Communicative friction)
8. Types of cyberbullying experienced/observed (Harassment typology)
9. Platform where incident occurred (Digital environment)
10. Frequency of cyberbullying encounters (Expo

## 6. Predictor Selection & Matrix Separation
Filter records with valid target responses and separate the dataset into predictor matrix $X$ and target vector $y$.

In [6]:
# Filter valid target records (N = 514)
valid_df = df_cleaned[df_cleaned[col_target].notna()].reset_index(drop=True)
print(f"Valid Analysis Dataset: {len(valid_df)} records")

# Target mapping
class_to_idx = {'Not at all': 0, 'Slightly': 1, 'Moderately': 2, 'Severely': 3}
y_full = valid_df[col_target].map(class_to_idx).values

# Map questionnaire column identifiers
col_age = [c for c in valid_df.columns if '1. What is your age?' in c][0]
col_gender = [c for c in valid_df.columns if '2. What is your gender?' in c][0]
col_plat = [c for c in valid_df.columns if '3. Which social media' in c][0]
col_usage = [c for c in valid_df.columns if c.startswith('4.')][0]
col_q5 = [c for c in valid_df.columns if '5. Have you personally experienced' in c][0]
col_q6 = [c for c in valid_df.columns if c.startswith('6.')][0]
col_q7 = [c for c in valid_df.columns if c.startswith('7.')][0]
col_q9 = [c for c in valid_df.columns if c.startswith('9.')][0]
col_q10 = [c for c in valid_df.columns if c.startswith('10.')][0]
col_q11 = [c for c in valid_df.columns if c.startswith('11.')][0]
col_q15 = [c for c in valid_df.columns if c.startswith('15.')][0]
col_q17 = [c for c in valid_df.columns if c.startswith('17.')][0]
col_q18 = [c for c in valid_df.columns if c.startswith('18.')][0]

retained_predictor_cols = [
    col_age, col_gender, col_plat, col_usage, col_q5, col_q6,
    col_q7, col_q9, col_q10, col_q11, col_q15, col_q17, col_q18
]

X_raw_df = valid_df[retained_predictor_cols].copy()
print(f"Retained Predictor Items: {len(retained_predictor_cols)} questionnaire variables")
print(f"Target Vector Length     : {len(y_full)} outcomes")
assert col_target not in X_raw_df.columns, "Target variable must not be present in X."
print("VERIFIED: X contains no target leakage.")

Valid Analysis Dataset: 514 records
Retained Predictor Items: 13 questionnaire variables
Target Vector Length     : 514 outcomes
VERIFIED: X contains no target leakage.


## 7. Ordinal Encoding
Encode survey questions that possess an intrinsic hierarchical or intensity ordering:
- **Age:** Below 18 (0) $\rightarrow$ 18–22 (1) $\rightarrow$ 23–30 (2) $\rightarrow$ Above 30 (3)
- **Daily Usage:** Less than 1 hr (0) $\rightarrow$ 1–3 hrs (1) $\rightarrow$ 3–5 hrs (2) $\rightarrow$ More than 5 hrs (3)
- **Encounter Frequency:** Never (0) $\rightarrow$ Rarely (1) $\rightarrow$ Sometimes (2) $\rightarrow$ Often (3) $\rightarrow$ Very Often (4)
- **Binary Exposure (Q5 & Q6):** No (0) $\rightarrow$ Yes (1)

In [7]:
age_map = {'Below 18': 0, '18–22': 1, '23–30': 2, 'Above 30': 3, '31–40': 3, '41–50': 4}
usage_map = {'Less than 1 hour': 0, '1–3 hours': 1, '3–5 hours': 2, 'More than 5 hours': 3}
freq_map = {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3, 'Very Often': 4}
bin_map = {'No': 0, 'Yes': 1}

print("Ordinal encoding dictionaries established:")
print(" - Age Map           :", age_map)
print(" - Usage Map         :", usage_map)
print(" - Frequency Map     :", freq_map)
print(" - Binary Map (Q5/Q6):", bin_map)

Ordinal encoding dictionaries established:
 - Age Map           : {'Below 18': 0, '18–22': 1, '23–30': 2, 'Above 30': 3, '31–40': 3, '41–50': 4}
 - Usage Map         : {'Less than 1 hour': 0, '1–3 hours': 1, '3–5 hours': 2, 'More than 5 hours': 3}
 - Frequency Map     : {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3, 'Very Often': 4}
 - Binary Map (Q5/Q6): {'No': 0, 'Yes': 1}


## 8. One-Hot Encoding
Encode single-choice nominal variables that possess no intrinsic mathematical order into binary indicator vectors:
- **Gender (Q2):** Female, Male, Non-binary, Prefer not to say
- **Posted Offensive Content (Q7):** No, Not Sure, Yes
- **Incident Platform (Q10):** Facebook, Instagram, Others, WhatsApp, YouTube
- **Context Area (Q17):** Family, Friends Circle, Online Gaming, Other, School / College, Social Media Community, Unknown Stranger, Workplace

In [8]:
nominal_cols = [
    ('Gender', col_gender),
    ('Posted_Offensive', col_q7),
    ('Incident_Platform', col_q10),
    ('Context_Area', col_q17)
]

for name, col in nominal_cols:
    cats = sorted(X_raw_df[col].dropna().unique())
    print(f"Nominal Variable: {name:<20} | Categories ({len(cats)}): {cats}")

Nominal Variable: Gender               | Categories (4): ['Female', 'Male', 'Non-binary', 'Prefer not to say']
Nominal Variable: Posted_Offensive     | Categories (3): ['No', 'Not Sure', 'Yes']
Nominal Variable: Incident_Platform    | Categories (5): ['Facebook', 'Instagram', 'Others', 'WhatsApp', 'YouTube']
Nominal Variable: Context_Area         | Categories (8): ['Family', 'Friends Circle', 'Online Gaming', 'Other', 'School / College', 'Social Media Community', 'Unknown Stranger', 'Workplace']


## 9. Multi-Hot Encoding
For multi-select questionnaire items where respondents can select multiple checkboxes, extract the vocabulary of individual tokens and represent each option as an independent binary (0/1) indicator feature:
- **Regular Platforms Used (Q3)**
- **Cyberbullying Types (Q9)**
- **Sought Help Channels (Q15)**
- **Actions Taken (Q18)**

In [9]:
def extract_tokens(series):
    tokens = set()
    for val in series.dropna():
        for t in str(val).split(','):
            ct = t.strip()
            if ct:
                tokens.add(ct)
    return sorted(list(tokens))

multiselect_items = [
    ('Platforms_Used (Q3)', col_plat),
    ('Bullying_Types (Q9)', col_q9),
    ('Sought_Help (Q15)', col_q15),
    ('Action_Taken (Q18)', col_q18)
]

for label, col in multiselect_items:
    toks = extract_tokens(X_raw_df[col])
    print(f"Multi-Select: {label:<22} | Unique Options ({len(toks)}): {toks}")

Multi-Select: Platforms_Used (Q3)    | Unique Options (6): ['Facebook', 'Instagram', 'Others', 'WhatsApp', 'X (Twitter)', 'YouTube']
Multi-Select: Bullying_Types (Q9)    | Unique Options (10): ['Body Shaming', 'Fake Profile', 'Fake Rumors', 'Hate Speech', 'Impersonation', 'Offensive Comments', 'Other', 'Sexual Harassment', 'Stalking', 'Threats']
Multi-Select: Sought_Help (Q15)      | Unique Options (7): ['Counselor', 'Family', 'Friends', 'Helpline', 'No', 'Psychologist', 'Teacher']
Multi-Select: Action_Taken (Q18)     | Unique Options (6): ['Blocked the user', 'Ignored it', 'Reported the account', 'Sought Professional Help', 'Told Friends / Family', 'Took No Action']


## 10. Generating the 54 Numeric Features
Integrate ordinal, one-hot, and multi-hot transformations into a unified feature engineering pipeline.  
Verify that the 13 survey questions transform into **exactly 54 numeric features**.

In [10]:
# Build feature encoding summary table
feature_summary_records = [
    {
        'Question_Number': 'Q1',
        'Survey_Question': 'What is your age category?',
        'Question_Type': 'Single-choice Ordinal',
        'Encoding_Method': 'Ordinal Encoding',
        'Generated_Features_Count': 1,
        'Feature_Names': 'Age_Ordinal'
    },
    {
        'Question_Number': 'Q2',
        'Survey_Question': 'What is your gender identity?',
        'Question_Type': 'Single-choice Nominal',
        'Encoding_Method': 'One-Hot Encoding',
        'Generated_Features_Count': 4,
        'Feature_Names': 'Gender_Female, Gender_Male, Gender_Non-binary, Gender_Prefer not to say'
    },
    {
        'Question_Number': 'Q3',
        'Survey_Question': 'Which social media platforms do you use regularly?',
        'Question_Type': 'Multi-choice (Checkboxes)',
        'Encoding_Method': 'Multi-Hot Encoding',
        'Generated_Features_Count': 6,
        'Feature_Names': 'Platform_Used_Facebook, Instagram, Others, WhatsApp, X (Twitter), YouTube'
    },
    {
        'Question_Number': 'Q4',
        'Survey_Question': 'How many hours do you spend on social media per day?',
        'Question_Type': 'Single-choice Ordinal',
        'Encoding_Method': 'Ordinal Encoding',
        'Generated_Features_Count': 1,
        'Feature_Names': 'Daily_Usage_Ordinal'
    },
    {
        'Question_Number': 'Q5',
        'Survey_Question': 'Have you personally experienced cyberbullying or harassment?',
        'Question_Type': 'Single-choice Binary',
        'Encoding_Method': 'Binary Indicator',
        'Generated_Features_Count': 1,
        'Feature_Names': 'Experienced_Cyberbullying_Binary'
    },
    {
        'Question_Number': 'Q6',
        'Survey_Question': 'Have you ever witnessed someone being cyberbullied online?',
        'Question_Type': 'Single-choice Binary',
        'Encoding_Method': 'Binary Indicator',
        'Generated_Features_Count': 1,
        'Feature_Names': 'Witnessed_Cyberbullying_Binary'
    },
    {
        'Question_Number': 'Q7',
        'Survey_Question': 'Have you ever posted a message that could have hurt someone?',
        'Question_Type': 'Single-choice Nominal',
        'Encoding_Method': 'One-Hot Encoding',
        'Generated_Features_Count': 3,
        'Feature_Names': 'Posted_Offensive_No, Posted_Offensive_Not Sure, Posted_Offensive_Yes'
    },
    {
        'Question_Number': 'Q9',
        'Survey_Question': 'What type of cyberbullying have you experienced or observed?',
        'Question_Type': 'Multi-choice (Checkboxes)',
        'Encoding_Method': 'Multi-Hot Encoding',
        'Generated_Features_Count': 10,
        'Feature_Names': 'Bullying_Type_Body Shaming, Fake Profile, Fake Rumors, Hate Speech, Impersonation, Offensive Comments, Other, Sexual Harassment, Stalking, Threats'
    },
    {
        'Question_Number': 'Q10',
        'Survey_Question': 'On which social media platform did the incident occur?',
        'Question_Type': 'Single-choice Nominal',
        'Encoding_Method': 'One-Hot Encoding',
        'Generated_Features_Count': 5,
        'Feature_Names': 'Incident_Platform_Facebook, Instagram, Others, WhatsApp, YouTube'
    },
    {
        'Question_Number': 'Q11',
        'Survey_Question': 'How often have you experienced cyberbullying online?',
        'Question_Type': 'Single-choice Ordinal',
        'Encoding_Method': 'Ordinal Encoding',
        'Generated_Features_Count': 1,
        'Feature_Names': 'Cyberbullying_Frequency_Ordinal'
    },
    {
        'Question_Number': 'Q15',
        'Survey_Question': 'Did you seek help from anyone?',
        'Question_Type': 'Multi-choice (Checkboxes)',
        'Encoding_Method': 'Multi-Hot Encoding',
        'Generated_Features_Count': 7,
        'Feature_Names': 'Sought_Help_Counselor, Family, Friends, Helpline, No, Psychologist, Teacher'
    },
    {
        'Question_Number': 'Q17',
        'Survey_Question': 'In which area or context did you experience cyberbullying?',
        'Question_Type': 'Single-choice Nominal',
        'Encoding_Method': 'One-Hot Encoding',
        'Generated_Features_Count': 8,
        'Feature_Names': 'Context_Area_Family, Friends Circle, Online Gaming, Other, School / College, Social Media Community, Unknown Stranger, Workplace'
    },
    {
        'Question_Number': 'Q18',
        'Survey_Question': 'What action did you take after experiencing cyberbullying?',
        'Question_Type': 'Multi-choice (Checkboxes)',
        'Encoding_Method': 'Multi-Hot Encoding',
        'Generated_Features_Count': 6,
        'Feature_Names': 'Action_Taken_Blocked the user, Ignored it, Reported the account, Sought Professional Help, Told Friends / Family, Took No Action'
    }
]

df_feature_summary = pd.DataFrame(feature_summary_records)
display(df_feature_summary[['Question_Number', 'Survey_Question', 'Question_Type', 'Encoding_Method', 'Generated_Features_Count']])
total_gen_features = df_feature_summary['Generated_Features_Count'].sum()
print(f"Total Retained Survey Questions: {len(df_feature_summary)}")
print(f"Total Generated Numeric Features: {total_gen_features}")
assert total_gen_features == 54, f"Feature count must be exactly 54. Got {total_gen_features}."
print("VERIFIED: 13 survey questionnaire items generate exactly 54 numeric predictor features.")

,Question_Number,Survey_Question,Question_Type,Encoding_Method,Generated_Features_Count
0,Q1,What is your age category?,Single-choice Ordinal,Ordinal Encoding,1
1,Q2,What is your gender identity?,Single-choice Nominal,One-Hot Encoding,4
2,Q3,Which social media platforms do you use regula...,Multi-choice (Checkboxes),Multi-Hot Encoding,6
3,Q4,How many hours do you spend on social media pe...,Single-choice Ordinal,Ordinal Encoding,1
4,Q5,Have you personally experienced cyberbullying ...,Single-choice Binary,Binary Indicator,1
5,Q6,Have you ever witnessed someone being cyberbul...,Single-choice Binary,Binary Indicator,1
6,Q7,Have you ever posted a message that could have...,Single-choice Nominal,One-Hot Encoding,3
7,Q9,What type of cyberbullying have you experience...,Multi-choice (Checkboxes),Multi-Hot Encoding,10
8,Q10,On which social media platform did the inciden...,Single-choice Nominal,One-Hot Encoding,5
9,Q11,How often have you experienced cyberbullying o...,Single-choice Ordinal,Ordinal Encoding,1


Total Retained Survey Questions: 13
Total Generated Numeric Features: 54
VERIFIED: 13 survey questionnaire items generate exactly 54 numeric predictor features.


## 11. Preprocessing Validation
Validate that the transformed feature matrix contains strictly numeric dtypes, no missing values, and zero target leakage.

In [11]:
# Import production SurveyFeaturePreprocessor
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from models.preprocessor import SurveyFeaturePreprocessor

preprocessor = SurveyFeaturePreprocessor(scenario='B')
preprocessor.fit(X_raw_df)
X_transformed = preprocessor.transform(X_raw_df)

print(f"Transformed Predictor Matrix Shape: {X_transformed.shape[0]} Rows × {X_transformed.shape[1]} Columns")
print("Data types check (all numeric):", np.issubdtype(X_transformed.values.dtype, np.number))
print("NaN / Missing values count   :", X_transformed.isna().sum().sum())
print("First 10 Feature Names:")
for i, fn in enumerate(preprocessor.feature_names_[:10]):
    print(f"  [{i+1:02d}] {fn}")
print(f"  ... and {len(preprocessor.feature_names_) - 10} more features (total: {len(preprocessor.feature_names_)}).")

Transformed Predictor Matrix Shape: 514 Rows × 54 Columns
Data types check (all numeric): True
NaN / Missing values count   : 0
First 10 Feature Names:
  [01] Age_Ordinal
  [02] Daily_Usage_Ordinal
  [03] Experienced_Cyberbullying_Binary
  [04] Witnessed_Cyberbullying_Binary
  [05] Cyberbullying_Frequency_Ordinal
  [06] Gender_Female
  [07] Gender_Male
  [08] Gender_Non-binary
  [09] Gender_Prefer not to say
  [10] Posted_Offensive_No
  ... and 44 more features (total: 54).


## 12. Stratified Train / Test Split
Partition the dataset into an **80% training set ($N = 411$)** and a **20% testing set ($N = 103$)** using stratified sampling on the four target classes (`random_state = 42`).

> **Data Snooping Prevention:** Encoders and feature vocabulary are fitted strictly on the training partition ($X_{train}$), ensuring that the test partition ($X_{test}$) remains completely unseen.

In [12]:
# 1. Stratified split on raw predictor DataFrame to prevent data leakage
train_idx, test_idx = train_test_split(
    np.arange(len(valid_df)),
    test_size=0.20,
    random_state=42,
    stratify=y_full
)

X_train_raw = X_raw_df.iloc[train_idx].reset_index(drop=True)
X_test_raw = X_raw_df.iloc[test_idx].reset_index(drop=True)
y_train = y_full[train_idx]
y_test = y_full[test_idx]

# 2. Fit preprocessor strictly on training data
train_preprocessor = SurveyFeaturePreprocessor(scenario='B')
train_preprocessor.fit(X_train_raw)

# 3. Transform training and test partitions
X_train = train_preprocessor.transform(X_train_raw)
X_test = train_preprocessor.transform(X_test_raw)

print(f"Training Matrix Shape : {X_train.shape[0]} Rows × {X_train.shape[1]} Columns")
print(f"Testing Matrix Shape  : {X_test.shape[0]} Rows × {X_test.shape[1]} Columns")
print(f"Training Outcomes (y) : {len(y_train)} records")
print(f"Testing Outcomes (y)  : {len(y_test)} records\n")

# Display class balance across splits
split_dist = pd.DataFrame({
    'Impact_Class': ordered_classes,
    'Full_Dataset': pd.Series(y_full).value_counts().sort_index().values,
    'Train_Count': pd.Series(y_train).value_counts().sort_index().values,
    'Train_Pct (%)': (pd.Series(y_train).value_counts().sort_index() / len(y_train) * 100).round(2).values,
    'Test_Count': pd.Series(y_test).value_counts().sort_index().values,
    'Test_Pct (%)': (pd.Series(y_test).value_counts().sort_index() / len(y_test) * 100).round(2).values
})
display(split_dist)

Training Matrix Shape : 411 Rows × 54 Columns


Testing Matrix Shape  : 103 Rows × 54 Columns
Training Outcomes (y) : 411 records
Testing Outcomes (y)  : 103 records



,Impact_Class,Full_Dataset,Train_Count,Train_Pct (%),Test_Count,Test_Pct (%)
0,Not at all,229,183,44.530,46,44.660
1,Slightly,137,109,26.520,28,27.180
2,Moderately,107,86,20.920,21,20.390
3,Severely,41,33,8.030,8,7.770


## 13. Final Preprocessing Summary
Master 12-point synthesis of the data cleaning, feature engineering, and validation process.

In [13]:
summary_table = [
    ('1. Raw Dataset Dimensions', f"{df_raw.shape[0]} rows × {df_raw.shape[1]} columns"),
    ('2. Cleaned Analytical Cohort', f"{len(df_cleaned)} rows (2 completely blank submissions removed)"),
    ('3. Single Target Variable', 'Mental_Health_Impact (Ordinal Multiclass)'),
    ('4. Number of Target Classes', '4 discrete classes: Not at all, Slightly, Moderately, Severely'),
    ('5. Target Leakage Prevention', 'Excluded Q13 (Negative Symptoms) and Q14 (Emotional Severity Rating)'),
    ('6. Retained Predictor Questionnaire Items', '13 survey questions (Demographics, Exposure, Platform, Frequency, Support, Action)'),
    ('7. Categorical Encoding Methods Applied', 'Ordinal Encoding (Ranked), One-Hot (Nominal), Multi-Hot (Checkboxes)'),
    ('8. Final Numeric Predictor Features', '54 numeric features generated from 13 questionnaire items'),
    ('9. Training Partition Size', f"{len(X_train)} records (80.0%)"),
    ('10. Testing Partition Size', f"{len(X_test)} records (20.0%)"),
    ('11. Stratified Split Random State', 'random_state = 42 (Stratified by 4 target classes)'),
    ('12. Data Snooping Prevention Status', 'CONFIRMED: Preprocessor fitted strictly on training partition; test set held out')
]

df_preprocessing_summary = pd.DataFrame(summary_table, columns=['Audit_Checkpoint', 'Specification_Finding'])
display(df_preprocessing_summary)

,Audit_Checkpoint,Specification_Finding
0,1. Raw Dataset Dimensions,521 rows × 19 columns
1,2. Cleaned Analytical Cohort,519 rows (2 completely blank submissions removed)
2,3. Single Target Variable,Mental_Health_Impact (Ordinal Multiclass)
3,4. Number of Target Classes,"4 discrete classes: Not at all, Slightly, Mode..."
4,5. Target Leakage Prevention,Excluded Q13 (Negative Symptoms) and Q14 (Emot...
5,6. Retained Predictor Questionnaire Items,"13 survey questions (Demographics, Exposure, P..."
6,7. Categorical Encoding Methods Applied,"Ordinal Encoding (Ranked), One-Hot (Nominal), ..."
7,8. Final Numeric Predictor Features,54 numeric features generated from 13 question...
8,9. Training Partition Size,411 records (80.0%)
9,10. Testing Partition Size,103 records (20.0%)


## 14. Export Processed Datasets & Summary Reports
Export standardized CSV outputs into `Data/processed/` and `results/`.

In [14]:
export_dirs = [
    os.path.join('..', 'Data', 'processed'),
    os.path.join('Data', 'processed'),
    os.path.join('..', 'results'),
    os.path.join('results')
]

valid_export_dirs = [d for d in export_dirs if os.path.exists(os.path.dirname(d)) or os.path.exists(d)]
for d in valid_export_dirs:
    os.makedirs(d, exist_ok=True)

# Datasets and tables to save
export_dict = {
    'cleaned_survey_data.csv': df_cleaned,
    'X_train.csv': X_train,
    'X_test.csv': X_test,
    'y_train.csv': pd.DataFrame({'Mental_Health_Impact_Class': y_train}),
    'y_test.csv': pd.DataFrame({'Mental_Health_Impact_Class': y_test}),
    'data_quality_report.csv': df_quality_report,
    'feature_encoding_summary.csv': df_feature_summary,
    'preprocessing_summary.csv': df_preprocessing_summary
}

for d in set(valid_export_dirs):
    for filename, df_out in export_dict.items():
        out_path = os.path.join(d, filename)
        df_out.to_csv(out_path, index=False, encoding='utf-8')

print("All preprocessed datasets and summary reports exported successfully.")

All preprocessed datasets and summary reports exported successfully.
